# 🚀 Demostración del Orquestador Jerárquico Multi-Agente con LangGraphEste Jupyter Notebook forma parte del proyecto Multi-Agent Orchestrator para Coderhouse.## 🎯 Objetivos de la Demostración1. **Demostrar la Arquitectura Jerárquica y Herramientas Exclusivas:** El Investigador y el Analista operan con responsabilidades desacopladas.2. **Enrutamiento Estructurado con Pydantic y Literal:** El nodo Supervisor utiliza `with_structured_output` y un esquema tipado (`RouterSchema`).3. **Prevención del 'Supervisor Infinito':** Control estricto de iteraciones máximas (`step_count >= 5`).

In [ ]:
# 1. Instalar dependencias necesarias# !pip install -q langgraph langchain-openai langchain-core pydantic python-dotenv

## 🛠️ Configuración de Estado y Contrato TipadoDefinimos el estado global (`TeamState`), el contrato del enrutador con `Literal` (`RouterSchema`) y un `MockLLM` compatible con `with_structured_output`.

In [ ]:
import os
from typing import TypedDict, Annotated, Optional, List, Literal
from pydantic import BaseModel, Field
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph.message import add_messages
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.outputs import ChatResult, ChatGeneration

class RouterSchema(BaseModel):
    next_step: Literal["Investigador", "Analista", "FINISH"] = Field(
        description="Próximo agente a invocar o 'FINISH' si la tarea está completada."
    )
    reasoning: str = Field(description="Justificación breve de la decisión.")

class TeamState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    supervisor_decision: Optional[RouterSchema]
    research_data: Optional[str]
    analysis_data: Optional[str]
    step_count: int

class MockLLM(BaseChatModel):
    structured_output_schema: Optional[type] = None
    def with_structured_output(self, schema):
        new_llm = MockLLM()
        new_llm.structured_output_schema = schema
        return new_llm
    def _generate(self, messages: list[BaseMessage], stop=None, run_manager=None, **kwargs) -> ChatResult:
        if self.structured_output_schema:
            has_res = any("Investigador" in str(m.content) for m in messages)
            has_ana = any("Analista" in str(m.content) for m in messages)
            if not has_res:
                dec = self.structured_output_schema(next_step="Investigador", reasoning="Falta investigar.")
            elif not has_ana:
                dec = self.structured_output_schema(next_step="Analista", reasoning="Falta analizar.")
            else:
                dec = self.structured_output_schema(next_step="FINISH", reasoning="Completado.")
            return ChatResult(generations=[ChatGeneration(message=AIMessage(content=dec.model_dump_json()))])
        return ChatResult(generations=[ChatGeneration(message=AIMessage(content="Simulación ejecutada correctamente."))])
    @property
    def _llm_type(self) -> str:
        return "mock_structured"

## 🧠 Definición de Nodos y Construcción del Grafo

In [ ]:
from langgraph.graph import StateGraph, START, JSONLoader, END

def supervisor_node(state: TeamState) -> dict:
    step_count = state.get("step_count", 0)
    if step_count >= 5:
        return {"supervisor_decision": RouterSchema(next_step="FINISH", reasoning="Límite alcanzado."), "step_count": step_count + 1}
    llm = MockLLM()
    structured_llm = llm.with_structured_output(RouterSchema)
    response = structured_llm.invoke([HumanMessage(content="Evaluar estado actual y decidir paso.")])
    decision = RouterSchema.model_validate_json(response.content) if isinstance(response.content, str) else response
    return {"supervisor_decision": decision, "step_count": step_count + 1}

def research_node(state: TeamState) -> dict:
    return {"messages": [HumanMessage(content="[INVESTIGADOR]: Datos de mercado recopilados.", name="Investigador")], "research_data": "Datos de mercado recopilados."}

def analyst_node(state: TeamState) -> dict:
    return {
        "messages": [HumanMessage(content="[ANALISTA]: Métricas procesadas con éxito.", name="Analista")],
        "analysis_data": "Métricas procesadas con éxito."
    }

builder = StateGraph(TeamState)
builder.add_node("Supervisor", supervisor_node)
builder.add_node("Investigador", research_node)
builder.add_node("Analista", analyst_node)
builder.set_entry_point("Supervisor")

def route_supervisor(state: TeamState) -> str:
    dec = state.get("supervisor_decision")
    return dec.next_step if hasattr(dec, "next_step") else "FINISH"

builder.add_conditional_edges("Supervisor", route_supervisor, {"Investigador": "Investigador", "Analista": "Analista", "FINISH": END})
builder.add_edge("Investigador", "Supervisor")
builder.add_edge("Analista", "Supervisor")
app = builder.compile()

## 🎬 Ejecución del Orquestador

In [ ]:
initial_state = {"messages": [HumanMessage(content="Iniciar análisis.")], "supervisor_decision": None, "research_data": None, "analysis_data": None, "step_count": 0}
for step in app.stream(initial_state):
    for node, upd in step.items():
        print(f"Nodo: {node} -> Actualización: {upd}")